In [1]:
# 1. Install dependencies
!pip install pmdarima -q

import joblib
import numpy as np
import requests
import warnings
from datetime import datetime, timedelta
from tensorflow.keras.models import load_model

warnings.filterwarnings('ignore')

# 2. Load the "Brains" into memory
print("Booting up AI Predictive Monitoring System...")
live_lstm = load_model('iocl_lstm_hybrid.keras')
live_arima = joblib.load('iocl_arima_hybrid.pkl')
live_scaler_X = joblib.load('iocl_scaler_X.pkl')
live_scaler_y = joblib.load('iocl_scaler_y.pkl')
print("✅ Models and Scalers Loaded Successfully! Ready for live inference.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 23.7 MB/s eta 0:00:00
Booting up AI Predictive Monitoring System...
✅ Models and Scalers Loaded Successfully! Ready for live inference.


In [3]:
API_KEY = "a2a83d4a9b876d70c4292be163c5ec2f"
IOCL_REFINERIES = {
    "panipat": ("Panipat Refinery", 29.4449, 76.8770),
    "mathura": ("Mathura Refinery", 27.3826, 77.6698)
}

# --- 1. AQI CALCULATION HELPERS ---
CPCB_BREAKPOINTS = {
    'pm2_5': [(0,30,0,50), (31,60,51,100), (61,90,101,150), (91,120,151,200), (121,250,201,300), (251,350,301,400), (351,430,401,500)],
    'pm10' : [(0,50,0,50), (51,100,51,100), (101,150,101,150), (151,250,151,200), (251,350,201,300), (351,430,301,400), (431,600,401,500)],
    'no2'  : [(0,40,0,50), (41,80,51,100), (81,180,101,150), (181,280,151,200), (281,400,201,300), (401,800,301,400), (801,1200,401,500)],
    'o3'   : [(0,50,0,50), (51,100,51,100), (101,168,101,150), (169,208,151,200), (209,748,201,300), (749,1000,301,400), (1001,1200,401,500)],
    'co'   : [(0,1.0,0,50), (1.1,2.0,51,100), (2.1,10,101,150), (10.1,17,151,200), (17.1,34,201,300), (34.1,50,301,400), (50.1,100,401,500)],
    'so2'  : [(0,40,0,50), (41,80,51,100), (81,380,101,150), (381,800,151,200), (801,1600,201,300), (1601,2000,301,400), (2001,2500,401,500)],
    'nh3'  : [(0,200,0,50), (201,400,51,100), (401,800,101,150), (801,1200,151,200), (1201,1800,201,300), (1801,2400,301,400), (2401,3000,401,500)]
}

def sub_index(c, bp_lo, bp_hi, i_lo, i_hi):
    if c <= bp_lo: return i_lo
    if c >= bp_hi: return i_hi
    return i_lo + (i_hi - i_lo) * (c - bp_lo) / (bp_hi - bp_lo)

def cpcb_sub_index(p, c):
    if p not in CPCB_BREAKPOINTS: return 0
    for bp in CPCB_BREAKPOINTS[p]:
        if c <= bp[1]: return sub_index(c, bp[0], bp[1], bp[2], bp[3])
    return sub_index(c, CPCB_BREAKPOINTS[p][-1][0], CPCB_BREAKPOINTS[p][-1][1], CPCB_BREAKPOINTS[p][-1][2], CPCB_BREAKPOINTS[p][-1][3])

def aqi_cpcb(concentrations):
    sub_idxs = {p: cpcb_sub_index(p, c) for p, c in concentrations.items()}
    dominant = max(sub_idxs, key=sub_idxs.get)
    return int(np.ceil(sub_idxs[dominant])), dominant.upper()


# --- 2. THE RECOMMENDATION ENGINE ---
def run_root_cause_recommendation_engine(aqi_series, dominant_series, city_name):
    ALERT_THRESHOLD = 250
    mitigation_playbook = {
        "SO2": {"severity": "CRITICAL", "diagnosis": "High Sulfur Dioxide emissions.", "action": "Inspect Sulfur Recovery Unit (SRU)."},
        "CO": {"severity": "CRITICAL", "diagnosis": "Incomplete combustion.", "action": "Increase excess air in boilers. Inspect flare stack."},
        "NO2": {"severity": "WARNING", "diagnosis": "High Nitrogen Oxide levels.", "action": "Optimize combustion temperature. Inspect Low-NOx burners."},
        "PM10": {"severity": "WARNING", "diagnosis": "Heavy particulate matter.", "action": "Activate water sprinklers. Check Electrostatic Precipitators."},
        "PM2_5": {"severity": "WARNING", "diagnosis": "Fine particulate matter.", "action": "Check filtration systems on exhaust stacks."},
        "O3": {"severity": "ALERT", "diagnosis": "Secondary Ozone levels.", "action": "Check for fugitive VOC leaks in storage tanks."},
        "NH3": {"severity": "ALERT", "diagnosis": "Ammonia slip detected.", "action": "Inspect Sour Water Stripper unit."}
    }

    peak_aqi = max(aqi_series)
    if peak_aqi < ALERT_THRESHOLD:
        print("\n" + "="*60)
        print("✅ SYSTEM STATUS: GREEN (NORMAL)")
        print(f"Peak predicted AQI is {peak_aqi} (Threshold: {ALERT_THRESHOLD}). No mitigation required.")
        print("="*60)
        return

    peak_hour = aqi_series.index(peak_aqi) + 1
    root_cause = dominant_series[peak_hour - 1].replace('.', '_')
    playbook = mitigation_playbook.get(root_cause, {"severity": "WARNING", "diagnosis": f"Spike in {root_cause}.", "action": "Site-wide sweep."})

    print("\n" + "!"*60)
    print(f"🚨 AUTOMATED PROACTIVE ALERT: {city_name.upper()} 🚨")
    print("!"*60)
    print(f"Subject: {playbook['severity']} - AQI Breach Projected in {peak_hour} Hours")
    print("-" * 60)
    print(f" > Time to Impact : {peak_hour} hours from now")
    print(f" > Projected AQI  : {peak_aqi} (Threshold: {ALERT_THRESHOLD})")
    print(f" > Root Cause     : {root_cause} dominated profile")
    print("-" * 60)
    print(f"DIAGNOSIS & PLAYBOOK:")
    print(f" > {playbook['diagnosis']} -> {playbook['action']}")
    print("!"*60)


# --- 3. EXECUTE LIVE HYBRID FORECAST ---
refinery = input("Enter refinery to monitor (e.g., panipat, mathura): ").strip().lower()
if refinery not in IOCL_REFINERIES:
    print("Refinery not found.")
else:
    name, lat, lon = IOCL_REFINERIES[refinery]
    print(f"\nFetching live sensor data for {name}...")

    url = f"http://api.openweathermap.org/data/2.5/air_pollution?lat={lat}&lon={lon}&appid={API_KEY}"
    data = requests.get(url).json()['list'][0]['components']

    # FIX 1: Only feed the 1 feature selected by your Random Forest (e.g., 'co')
    live_vals = np.array([data['co']])
    scaled_live = live_scaler_X.transform(live_vals.reshape(1, -1))[0]

    print("Running 24-Hour Hybrid Forecast (ARIMA + LSTM)...")

    # Calculate true current AQI to feed into ARIMA
    current_aqi, _ = aqi_cpcb(data)
    live_arima.update([current_aqi])
    arima_preds = np.asarray(live_arima.predict(n_periods=24))

    # LSTM Non-Linear Forecast
    input_seq = np.tile(scaled_live, (24, 1))
    lstm_preds_scaled = []
    cur = input_seq.copy()
    for _ in range(24):
        # FIX 2: Reshape to (1, 24, 1) to match the single feature
        pred = live_lstm.predict(cur.reshape(1, 24, 1), verbose=0)[0]
        lstm_preds_scaled.append(pred)
        cur = np.roll(cur, -1, axis=0)
        cur[-1] = pred

    lstm_residuals = live_scaler_y.inverse_transform(np.array(lstm_preds_scaled)).flatten()

    # Combine and Extract AQI
    final_hybrid_target = np.maximum(arima_preds + lstm_residuals, 0)

    # Create realistic pollutant distributions based on the hybrid target
    forecast_pollutants = []
    for h in range(24):
        # Prevent division by zero if current_aqi is somehow 0
        multiplier = final_hybrid_target[h] / max(1, current_aqi)
        f_dict = {
            'co': data['co'] * multiplier, 'no': data['no'] * multiplier, 'no2': data['no2'] * multiplier,
            'o3': data['o3'] * multiplier, 'so2': data['so2'] * multiplier, 'pm2_5': data['pm2_5'] * multiplier,
            'pm10': data['pm10'] * multiplier, 'nh3': data['nh3'] * multiplier
        }
        forecast_pollutants.append(f_dict)

    # Calculate Final AQI
    aqi_series, dom_series = [], []
    for f in forecast_pollutants:
        aqi, dom = aqi_cpcb(f)
        aqi_series.append(aqi)
        dom_series.append(dom)

    print(f"\nForecast Complete. Current AQI: {aqi_series[0]}")

    # TRIGGER THE INNOVATION ENGINE
    run_root_cause_recommendation_engine(aqi_series, dom_series, name)

Enter refinery to monitor (e.g., panipat, mathura): panipat

Fetching live sensor data for Panipat Refinery...
Running 24-Hour Hybrid Forecast (ARIMA + LSTM)...

Forecast Complete. Current AQI: 500

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
🚨 AUTOMATED PROACTIVE ALERT: PANIPAT REFINERY 🚨
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Subject: CRITICAL - AQI Breach Projected in 1 Hours
------------------------------------------------------------
 > Time to Impact : 1 hours from now
 > Projected AQI  : 500 (Threshold: 250)
 > Root Cause     : CO dominated profile
------------------------------------------------------------
DIAGNOSIS & PLAYBOOK:
 > Incomplete combustion. -> Increase excess air in boilers. Inspect flare stack.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
